# Sommelier — Vietnamese smoke test on Google Colab

Runs the [sommelier](https://github.com/tuanad121/sommelier) podcast pipeline on **one** Vietnamese audio clip.

**ASR MoE (VN):** Whisper-large-v3 + PhoWhisper-large + ChunkFormer-CTC, ROVER-voted.

**Recommended runtime:** Colab Pro **L4 (24 GB)** or **A100 (40 GB)**. Free T4 (16 GB) may OOM with all three ASR models held in VRAM simultaneously — see the troubleshooting cell at the bottom.

**Before you run:** Runtime → Change runtime type → GPU.

**You will need:** a Hugging Face token with access accepted for `pyannote/segmentation-3.0`, `pyannote/embedding`, and `nvidia/diar_sortformer_4spk-v1`.

## 1. Sanity check the runtime

In [ ]:
!nvidia-smi -L
!python --version
!df -h /content | tail -1
import torch
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'avail', torch.cuda.is_available())

## 2. Clone the repo

If you're iterating on the `--lang vi` integration, swap in your fork's URL.

In [ ]:
%cd /content
![ -d sommelier ] || git clone https://github.com/tuanad121/sommelier.git
%cd /content/sommelier/podcast-pipeline

## 3. Install dependencies (~15 min)

Mirrors the README's three-step install order (torch first, then requirements, then re-pin torch). Adds the `chunkformer` package on top for the VN MoE third slot.

In [ ]:
# 1) PyTorch first (matches README)
!pip install -q torch==2.7.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu126

# 2) Main pinned requirements
!pip install -q -r requirements.txt

# 3) Re-pin torch (nemo-toolkit[all] often clobbers it)
!pip install -q torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu126

# 4) VN MoE extra: ChunkFormer
!pip install -q chunkformer

In [ ]:
# Verify imports — restart runtime if any of these fail
import torch, whisperx, demucs, nemo, pyannote.audio, transformers
from chunkformer import ChunkFormerModel  # noqa: F401
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
print('whisperx', whisperx.__version__)
print('transformers', transformers.__version__)
print('nemo', nemo.__version__)

## 4. Hugging Face authentication

Pyannote diarization and Sortformer are gated. Accept the license on each model page first, then paste your token below.

In [ ]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass('HF token (hf_...): ')
login(token=hf_token)

# Also write it to config.json so pipeline code that reads cfg['huggingface_token'] works.
import json, pathlib
cfg_path = pathlib.Path('/content/sommelier/podcast-pipeline/config.json')
cfg = json.loads(cfg_path.read_text())
cfg['huggingface_token'] = hf_token
cfg_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
print('config.json updated')

## 5. Provide a Vietnamese audio clip

Upload one short (~30–60 s) WAV/MP3. The pipeline expects a **folder** of audio, so we drop the file into `/content/vi_audio/`.

In [ ]:
import os, shutil
from google.colab import files

os.makedirs('/content/vi_audio', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f'/content/vi_audio/{name}')
!ls -la /content/vi_audio/

## 6. Run the pipeline (VN MoE, no Demucs/SepReformer for speed)

Flags below disable Demucs, SepReformer, Qwen3-Omni captioning, and the LLM post-pass to keep the smoke test fast and minimize VRAM. Add them back once the basic VN flow works.

In [ ]:
%cd /content/sommelier/podcast-pipeline
!python main_original_ASR_MoE.py \
  --input_folder_path /content/vi_audio \
  --lang vi \
  --vad \
  --dia3 \
  --ASRMoE \
  --no-demucs \
  --whisperx_word_timestamps \
  --no-qwen3omni \
  --no-sepreformer \
  --LLM case_0 \
  --seg_th 0.11 \
  --min_cluster_size 11 \
  --clust_th 0.5 \
  --merge_gap 2

## 7. Inspect output

The script writes per-clip outputs under `/content/vi_audio/_final/_processed_llm-twelve-cases-*/<audio_name>/`.

In [ ]:
import glob, json, pathlib

json_paths = sorted(glob.glob('/content/vi_audio/_final/**/*.json', recursive=True))
print(f'Found {len(json_paths)} result file(s):')
for p in json_paths:
    print(' -', p)

if json_paths:
    result = json.loads(pathlib.Path(json_paths[0]).read_text())
    print('\nMetadata:')
    print(json.dumps(result.get('metadata', {}), indent=2, ensure_ascii=False))
    print(f"\nFirst 3 segments of {len(result.get('segments', []))}:")
    for seg in result.get('segments', [])[:3]:
        print(json.dumps(seg, indent=2, ensure_ascii=False))

## Troubleshooting

**T4 OOM during MoE:** the three ASR models (Whisper-large-v3, PhoWhisper-large, ChunkFormer-CTC) plus Sortformer/pyannote barely fit in 16 GB. Either upgrade the runtime, or as a stopgap edit `asr_MoE()` in `main_original_ASR_MoE.py` to use `ThreadPoolExecutor(max_workers=1)` so models run sequentially instead of in parallel.

**`chunkformer` import error:** the package was added in step 3.4 above — if you skipped it, run `!pip install chunkformer` and **restart the runtime** (Runtime → Restart runtime).

**HF 401 on Sortformer / pyannote:** accept the license on each gated model page (`pyannote/segmentation-3.0`, `pyannote/embedding`, `nvidia/diar_sortformer_4spk-v1`) while logged in to the same HF account whose token you pasted.

**Empty `text_phowhisper` / `text_chunkformer` but `text_whisper` populated:** check the cell output for `PhoWhisper failed:` / `ChunkFormer failed:` log lines — the MoE swallows per-model errors so Whisper alone still produces a transcript.